# Cholesky Decomposition & Correlated Normals

This notebook covers three things:

1. **Cholesky decomposition from scratch**: factor a symmetric positive-definite matrix $A$ as $A = LL^T$.
2. **Simulating correlated normal random variables** using the Cholesky factor of a target correlation matrix.
3. **Repairing an invalid correlation matrix**: when a "correlation matrix" is not positive semi-definite, Cholesky fails. We fix it by clipping negative eigenvalues and rescaling.

In [3]:
import numpy as np

## 1. Cholesky decomposition from scratch

For a symmetric positive-definite matrix $A$, there is a unique lower-triangular matrix $L$ with positive diagonal such that

$$A = LL^T$$

Its entries can be computed row by row:

$$L_{ii} = \sqrt{A_{ii} - \sum_{k=1}^{i-1} L_{ik}^2}$$

$$L_{ij} = \frac{1}{L_{jj}}\left(A_{ij} - \sum_{k=1}^{j-1} L_{ik}L_{jk}\right), \quad j < i$$

Each entry depends only on entries already computed above it or to its left, so a double loop over $i$ and $j \le i$ is enough.

In [2]:
def cholesky(A):
    n=len(A)
    L=np.zeros((n,n))
    for i in range(n):
        for j in range(i+1):
            sum=[L[i][k]*L[j][k] for k in range(j)]
            if i==j:
                L[i,j]=(A[i,j]-sum)**0.5
            else:
                L[i,j]=(A[i,j]-sum)/L[j,j]

    return L

## 2. Simulating correlated normals with Cholesky

Suppose we want random variables $X = (X_1, \dots, X_n)$ with a target correlation matrix $R$.

1. Compute the Cholesky factor $L$ such that $R = LL^T$.
2. Draw independent standard normals $\varepsilon \sim \mathcal{N}(0, I)$.
3. Set $X = L\varepsilon$.

Then $\operatorname{Cov}(X) = L\,\operatorname{Cov}(\varepsilon)\,L^T = LIL^T = R$.

Below, each column of `eps` is one sample of $\varepsilon$, so `X = L @ eps` transforms all $N$ samples at once. With a large sample size, the empirical correlation matrix is very close to $R$.

In [4]:
import numpy as np

# target correlation matrix
R = np.array([
    [1.0, 0.6, 0.2],
    [0.6, 1.0, 0.5],
    [0.2, 0.5, 1.0]
])

# Cholesky factor of target correlation matrix
L = np.linalg.cholesky(R)

# simulate correlated normals
np.random.seed(0)
N = 100000
eps = np.random.randn(3, N)
X = L @ eps

# empirical correlation matrix
C = np.corrcoef(X)
print("Target R:\n", np.round(R, 3))
print("\nEmpirical correlation:\n", np.round(C, 3))
print("\nDifference:\n", np.round(C - R, 3))


Target R:
 [[1.  0.6 0.2]
 [0.6 1.  0.5]
 [0.2 0.5 1. ]]

Empirical correlation:
 [[1.    0.6   0.204]
 [0.6   1.    0.5  ]
 [0.204 0.5   1.   ]]

Difference:
 [[ 0.    -0.     0.004]
 [-0.     0.    -0.   ]
 [ 0.004 -0.    -0.   ]]


## 3. Repairing an invalid correlation matrix

Not every symmetric matrix with ones on the diagonal is a valid correlation matrix. It must also be **positive semi-definite** (all eigenvalues $\ge 0$).

Consider the matrix below: $X_1$ is strongly positively correlated with both $X_2$ and $X_3$, yet $X_2$ and $X_3$ are strongly *negatively* correlated with each other. These pairwise correlations are mutually inconsistent, so the matrix has a negative eigenvalue and `np.linalg.cholesky` would fail on it.

In [17]:
C = np.array([
    [1.0,  0.9,  0.9],
    [0.9,  1.0, -0.9],
    [0.9, -0.9,  1.0]
])

### Step 1: Eigendecomposition

Since $C$ is symmetric, we can write $C = Q \Lambda Q^T$, where $Q$ holds orthonormal eigenvectors and $\Lambda$ is the diagonal matrix of eigenvalues. `np.linalg.eigh` is the routine for symmetric matrices. Here the eigenvalues are $\{-0.8,\ 1.9,\ 1.9\}$; the $-0.8$ is what makes $C$ invalid.

In [18]:
eigenvalues,eigenvectors=np.linalg.eigh(C)

### Step 2: Clip negative eigenvalues

Replace every eigenvalue below a small threshold with $\varepsilon = 10^{-8}$ and rebuild the matrix:

$$\tilde{C} = Q\, \max(\Lambda, \varepsilon)\, Q^T$$

$\tilde{C}$ is now positive (semi-)definite, but its diagonal is no longer exactly 1, so it is a covariance matrix rather than a correlation matrix.

In [26]:
balanced=np.maximum(1e-8,eigenvalues)
cov=eigenvectors @ np.diag(balanced) @ eigenvectors.T


### Step 3: Rescale back to a correlation matrix

Let $d_i = \sqrt{\tilde{C}_{ii}}$ be the implied standard deviations. Dividing each entry by $d_i d_j$ restores a unit diagonal:

$$R_{ij} = \frac{\tilde{C}_{ij}}{d_i\, d_j} \qquad\text{i.e.}\qquad R = D^{-1}\tilde{C}D^{-1},\ \ D = \operatorname{diag}(d)$$

`np.outer(d, d)` builds the matrix of products $d_i d_j$ so the division can be done element-wise.

In [27]:
d=np.sqrt(np.diag(cov))

In [28]:
d

array([1.12546287, 1.12546287, 1.12546287])

In [29]:
corr=cov/np.outer(d,d)
print(corr)

[[ 1.   0.5  0.5]
 [ 0.5  1.  -0.5]
 [ 0.5 -0.5  1. ]]


The repaired matrix keeps the original **sign pattern** ($X_1$ positively correlated with $X_2$ and $X_3$, while $X_2$ and $X_3$ are negatively correlated), but the magnitudes shrink from $0.9$ to $0.5$, which is as strong as these relationships can be while staying consistent. This matrix is positive semi-definite and can now be fed into the Cholesky-based simulation from Section 2.

> **Note:** eigenvalue clipping is a quick, simple repair. It is not guaranteed to give the *closest* valid correlation matrix; for that, see Higham's (2002) nearest-correlation-matrix algorithm.

In [30]:
np.outer(d,d)

array([[1.26666667, 1.26666667, 1.26666667],
       [1.26666667, 1.26666667, 1.26666667],
       [1.26666667, 1.26666667, 1.26666667]])